In [7]:
import os
import numpy as np
import pandas as pd

In [3]:
FT_result_dir = os.path.join("..","finetune_pinnacle","tmp_model_outputs","pinnacle_seed=1")
csv_filenames = [i for i in os.listdir(FT_result_dir) if "_all_" in i]
csv_filenames

['pinnacle_all_preds_acinar cell of salivary gland.csv',
 'pinnacle_all_preds_adipocyte.csv',
 'pinnacle_all_preds_adventitial cell.csv',
 'pinnacle_all_preds_alveolar fibroblast.csv',
 'pinnacle_all_preds_artery endothelial cell.csv',
 'pinnacle_all_preds_b cell.csv',
 'pinnacle_all_preds_basal cell.csv',
 'pinnacle_all_preds_basal cell of prostate epithelium.csv',
 'pinnacle_all_preds_bladder urothelial cell.csv',
 'pinnacle_all_preds_bronchial smooth muscle cell.csv',
 'pinnacle_all_preds_bronchial vessel endothelial cell.csv',
 'pinnacle_all_preds_capillary endothelial cell.csv',
 'pinnacle_all_preds_cardiac endothelial cell.csv',
 'pinnacle_all_preds_cardiac muscle cell.csv',
 'pinnacle_all_preds_cd141-positive myeloid dendritic cell.csv',
 'pinnacle_all_preds_cd1c-positive myeloid dendritic cell.csv',
 'pinnacle_all_preds_cd24 neutrophil.csv',
 'pinnacle_all_preds_cd4-positive helper t cell.csv',
 'pinnacle_all_preds_cd4-positive, alpha-beta memory t cell.csv',
 'pinnacle_all_pre

In [4]:
df = pd.read_csv(os.path.join(FT_result_dir, csv_filenames[0]))
df.head()

,y,preds,name,type
0,0.0,0.514637,SHMT1,test
1,0.0,0.504812,AZGP1,val
2,0.0,0.248804,ALDH3A2,test
3,0.0,0.224450,GLRX,test
4,0.0,0.185788,KRT7,val


In [14]:
def gene_pred_extractor(csv_filename, gene_symbol="CDK6", BASE_PATH = FT_result_dir):
    df = pd.read_csv(os.path.join(BASE_PATH, csv_filename))
    cell_type_name = csv_filename.replace(".csv","").replace("pinnacle_all_preds_","")
    if gene_symbol in df["name"].values:
        score = df[df["name"] == gene_symbol]["preds"].values[0]
        return score, cell_type_name
    ## Cell does not have the gene
    else:
        return np.nan, cell_type_name

## use gene_pred_extractor with a custom gene_symbol and apply it to csv_filenames list. use map or apply to speed up things

gene_symbol = "JAK3"
results = list(map(lambda x: gene_pred_extractor(x, gene_symbol=gene_symbol), csv_filenames))
# convert the list which returns a tuple to a dataframe so that we can sort it
df_results = pd.DataFrame(results, columns=["score", "cell_type"])
## Sort by score in descending order
df_results = df_results.sort_values(by="score", ascending=False)
df_results.dropna(inplace=True)
df_results.reset_index(drop=True, inplace=True)
df_results.head()

,score,cell_type
0,0.235204,microglial cell
1,0.217462,hematopoietic stem cell
2,0.205055,granulocyte
3,0.181224,naive regulatory t cell
4,0.181009,nampt neutrophil


In [16]:
## Iterate through the filenames, load the CSV, add a column where the values will be the celltype name from the filename. Concatenate all the csvs
dfs = []
for csv_filename in csv_filenames:
    df = pd.read_csv(os.path.join(FT_result_dir, csv_filename))
    cell_type_name = csv_filename.replace(".csv","").replace("pinnacle_all_preds_","")
    df["cell_type"] = cell_type_name
    dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)

In [17]:
df_all.head()

,y,preds,name,type,cell_type
0,0.0,0.514637,SHMT1,test,acinar cell of salivary gland
1,0.0,0.504812,AZGP1,val,acinar cell of salivary gland
2,0.0,0.248804,ALDH3A2,test,acinar cell of salivary gland
3,0.0,0.224450,GLRX,test,acinar cell of salivary gland
4,0.0,0.185788,KRT7,val,acinar cell of salivary gland


In [19]:
## Groupby the name, for each groupped dataframe, find the row with greatest preds. make a df with these rows alone
df_best = df_all.loc[df_all.groupby("name")["preds"].idxmax()]
df_best = df_best.sort_values(by="preds", ascending=False)
df_best.head(20)

,y,preds,name,type,cell_type
8400,1.0,0.544774,FGFR3,train,conjunctival epithelial cell
24030,0.0,0.529834,TNK1,test,large intestine goblet cell
24031,1.0,0.527707,MAPKAPK5,train,large intestine goblet cell
0,0.0,0.514637,SHMT1,test,acinar cell of salivary gland
1,0.0,0.504812,AZGP1,val,acinar cell of salivary gland
47574,1.0,0.497695,PRIM2,train,type i nk t cell
41710,0.0,0.479576,ALAS1,train,retinal ganglion cell
42613,0.0,0.465789,GJA1,train,schwann cell
16440,1.0,0.458395,POLA2,test,fibroblast of cardiac tissue
2919,1.0,0.455851,JAK2,train,bronchial smooth muscle cell
